<a href="https://colab.research.google.com/github/akansalbe23-star/DeepLearningLab/blob/main/DL_Assignment_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# reading the Dataset
import pandas as pd
df=pd.read_csv("/content/glass.csv")

In [4]:
# finding the Number of Rows and columns
df.shape

(214, 10)

In [6]:
# Seeing the Column names
df.columns

Index(['RI', 'Na', 'Mg', 'Al', 'Si', 'K', 'Ca', 'Ba', 'Fe', 'Type'], dtype='object')

In [7]:
# Looking at the first few row
df.head()

,RI,Na,Mg,Al,Si,K,Ca,Ba,Fe,Type
0,1.52101,13.64,4.49,1.10,71.78,0.06,8.75,0.0,0.0,1
1,1.51761,13.89,3.60,1.36,72.73,0.48,7.83,0.0,0.0,1
2,1.51618,13.53,3.55,1.54,72.99,0.39,7.78,0.0,0.0,1
3,1.51766,13.21,3.69,1.29,72.61,0.57,8.22,0.0,0.0,1
4,1.51742,13.27,3.62,1.24,73.08,0.55,8.07,0.0,0.0,1


In [10]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 214 entries, 0 to 213
Data columns (total 10 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   RI      214 non-null    float64
 1   Na      214 non-null    float64
 2   Mg      214 non-null    float64
 3   Al      214 non-null    float64
 4   Si      214 non-null    float64
 5   K       214 non-null    float64
 6   Ca      214 non-null    float64
 7   Ba      214 non-null    float64
 8   Fe      214 non-null    float64
 9   Type    214 non-null    int64  
dtypes: float64(9), int64(1)
memory usage: 16.8 KB


In [11]:
# Q Which column is the output we want to predict?
# Ans Type
# Are all columns numeric?
# Ans Yes
# Is there an ID column that should not be used?
# Ans There is no such column

In [13]:
# We are converting our problem into binary problem
# Where type 1 = positive class , rest = Negative class
df["y"]=(df["Type"]==1).astype(int)
df=df.drop(columns=["Type"])


In [15]:
# Separating the input and Output features
X = df.drop(columns=["y"]).values
y = df["y"].values

In [16]:
# splitting the data into training and testing because a model cannot be tested using the training data , it will give false confidence
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X,y,test_size=0.2,random_state=42
)

In [17]:
# since the range of all the features are different we need to scale it to so no feature dominates the other
from sklearn.preprocessing import StandardScaler
scaler=StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [18]:
import numpy as np
def sigmoid(z):
  return (1/(1+np.exp(-z)))
# to give confidence values instead of decision i.e 'yes' or 'no' we make use of sigmoid function that gives us probabilistic values

In [19]:
def predict_prob(X,w,b):
  z=X @ w +b
  p=sigmoid(z)
  return p
# depending on the evidence score we find the confidence value

In [20]:
def loss(y,p):
  return -np.mean(y*np.log(p) + (1-y)*np.log(1-p))

# since there is no prediction here , and we are involved with confidence value , we do not see the correctness of the prediction.
# rather we see the fault in the confidence value , giving more penalty to confident wrong prediction

In [21]:
def update_weights(X,y,w,b,lr):
  p=predict_prob(X,w,b)
  error=p-y
  w=w-lr*(X.T @ error )/len(y)
  b=b-lr*np.mean(error)
  return w,b
#since we need to train the model to give confident correct prediction , we have to make sure it learns from its mistakes at a rate of lr

In [22]:
w=np.zeros(X_train.shape[1])
b=0.0
lr=0.1
epochs=100
for i in range(epochs):
  w,b=update_weights(X_train,y_train,w,b,lr)
# we train the model starting with weights and bias as 0 , as the model learns the weights and biases improves


In [23]:
def predict_label(p,threshold=0.5):
  return (p>=threshold).astype(int)

# we have set a threshold value and depending on that ,the confidence value is used to predict the type


In [25]:
p=predict_prob(X_test,w,b)
prediction=predict_label(p)
l=loss(y_test,p)
print(prediction,y_test)
print(l)
#I tested taking thresholds 0.5,0.7, 0.9 all of them gave loss approximately the same

[0 0 1 0 0 0 1 0 0 0 0 0 1 0 0 0 0 1 0 0 0 0 0 0 0 0 0 1 0 1 0 0 0 0 1 0 0
 1 0 0 0 0 1] [1 0 1 0 0 0 1 0 0 0 0 0 0 0 0 0 0 1 1 0 0 0 0 0 0 0 1 1 0 1 1 0 0 0 1 0 0
 0 0 0 0 0 1]
0.4108266473885576


# How it differs from perceptron ?
This differs from a perceptron, which uses a step function to make a decision; here, a sigmoid function provides the confidence of the output, showing how close it is to the correct value.

# Why sigmoid matters ?
Sigmoid is important because it provides the confidence of a prediction. For example, an output of 0.4 indicates the model is 40% confident that the prediction is correct.

# What problem still remains unsolved?
Since we converted the problem into a binary one—treating type 1 glass as the positive class and all others as negative—we haven’t addressed predicting the other glass types